# Frozen model validation exports

Purpose: infer with all six final ViT and learned kernel EMA/probe pairs on 8,074 previously unused Food-101 development images. Inputs are immutable Notebook 11 checkpoints, fixed epoch 20 probes, original image audit and split. Outputs are registered protocol, exact resume smoke, progress checkpoints, predictions and measured runtime. No training or official test inference.

Import only this `ahr-export` cell through `TaggedNotebookLoader`; it defines APIs without launching inference. Expected full inference is calibrated on existing development images and capped at 45 minutes. Checkpoints: `checkpoints/<out-directory-basename>/`, latest two hash-verified feature prefixes per model. Exact resume: call `run_validation(ROOT, OUT)` with identical protocol and unchanged sources before the original persisted deadline. Invalid manifests raise; deadlines never reset.

In [ ]:
"""Frozen final-1800 EMA/fixedepoch20 probes on unused development images.
Only tagged notebook definitions load. No training, model selection or official test.
"""
from pathlib import Path
import copy, csv, gc, hashlib, importlib.util, json, math, os, random, sys, time, uuid
import numpy as np
import torch
from torch.nn import functional as F

ROOT=Path(__file__).resolve().parents[2]
_boot={"__file__":str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for _cell in json.loads((ROOT/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:
    if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
        exec(compile(''.join(_cell['source']),str(ROOT/'fmm_jepa/bootstrap.ipynb'),'exec'),_boot)
_boot['install_notebook_package'](ROOT)
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
from fmm_jepa.food101_data import Food101Adapter, EvaluationDataset, preprocessing_descriptor, _read_sealed, canonical_hash
from fmm_jepa.food101_evaluation import (atomic_torch, encoder_state_hash, classification_metrics,
    prediction_rows, write_csv, validate_feature_cache, frozen_encoder)
from fmm_jepa.semantic_pilot import configure_environment

SEEDS=list(range(20261401,20261407))
VARIANTS=['vit','learned_kernel']
BATCH=64
CAP_SECONDS=2700

def load_tagged(root,name,folder='11_manifold_long',prefix='fmm11_'):
    key=prefix+name
    if key not in sys.modules:
        path=Path(root)/'experiments'/folder/(name+'.ipynb')
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(path))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def export_digest(path):
    cells=json.loads(Path(path).read_text())['cells']
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in cells
        if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()

def immutable(path,value):
    path=Path(path)
    if path.exists():
        if json.loads(path.read_text())!=value:raise ValueError('Immutable protocol/evidence changed: '+str(path))
    else:atomic_json(path,value)
    return value

def record(path):
    path=Path(path).resolve()
    return {'path':str(path),'sha256':digest_file(path)}

def verify(rec):
    path=Path(rec['path'])
    if digest_file(path)!=rec['sha256']:raise ValueError('Pinned source hash changed: '+str(path))
    return path

def _paths(root,out):
    root,out=Path(root).resolve(),Path(out).resolve();out.relative_to(root)
    run_id=out.name
    if not run_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in run_id):
        raise ValueError('Run ID must be a simple directory basename')
    return root,out,root/'checkpoints'/run_id

def _source_stage(root):
    return Path(root)/'artifacts/notebook_11/manifold-long-001/stages/learning'

def prepare_validation(root,out):
    """Freeze membership, source code, final checkpoints and final probes before predictions.
    Verifies every new image's encoded bytes against the original RGB audit; test pixels stay unread.
    """
    root,out,checkpoint=_paths(root,out);out.mkdir(parents=True,exist_ok=True)
    stage=_source_stage(root);plan=json.loads((stage/'plan.json').read_text())
    runner=load_tagged(root,'runner');runner.stage_definition_hashes(root)
    evaluator=load_tagged(root,'evaluation');evaluator._verify_plan_sources(root,plan)
    source_root=root/'artifacts/notebook_02/food101-pilot-001'
    split=json.loads((source_root/'split_manifest.json').read_text())
    audit=json.loads((source_root/'duplicate_audit.json').read_text())
    if canonical_hash({k:v for k,v in split.items()if k!='split_sha256'})!=split['split_sha256']:
        raise ValueError('Original split checksum changed')
    if canonical_hash({k:v for k,v in audit.items()if k!='audit_sha256'})!=audit['audit_sha256']:
        raise ValueError('Original decoded audit checksum changed')
    if split['duplicate_audit_sha256']!=audit['audit_sha256']:raise ValueError('Split/audit mismatch')
    source=_read_sealed(root/'data/food101/dataset_manifest.json')
    dataset=Food101Adapter(root/'data/food101',source)
    selected=set(split['selected_ids']['development'])
    fresh=[sid for sid in split['pool_ids']['development']if sid not in selected]
    heldout=set(split['pool_ids']['heldout']);train=set(split['pool_ids']['training'])
    if len(fresh)!=8074 or len(selected)!=2020 or len(heldout)!=25246 or len(set(fresh))!=len(fresh):
        raise ValueError('Expected 8074 reserve,2020 observed development and25246 reserved test')
    if set(fresh)&(selected|heldout|train|set(split['diagnostic_ids'])):
        raise ValueError('Fresh validation overlaps training,observed development,diagnostics or test')
    excluded={x['sample_id']for x in split['exclusions']}
    if set(fresh)&excluded:raise ValueError('Excluded original RGB duplicate reintroduced')
    expected=set(fresh);encoded={}
    for shard in audit['original_record_shards']:
        path=root/'data/food101/audit_shards'/shard['file']
        if digest_file(path)!=shard['sha256']:raise ValueError('Original audit shard changed')
        for entry in json.loads(path.read_text())['records']:
            sid=entry['sample_id']
            if sid in expected:
                image=dataset.path_for(sid)
                if image.stat().st_size!=entry['encoded_bytes']or digest_file(image)!=entry['encoded_sha256']:
                    raise ValueError('Validation image encoded bytes changed: '+sid)
                encoded[sid]=entry['encoded_sha256']
    if set(encoded)!=expected:raise ValueError('Missing original image audit records')
    arms=[]
    for seed in SEEDS:
        for variant in VARIANTS:
            model_id=f's{seed}-{variant}-u1800';summary_path=stage/'evaluation'/(model_id+'.json')
            saved=json.loads(summary_path.read_text())
            arm=next(a for a in plan['arms']if a['seed']==seed and a['variant']==variant)
            if saved['updates']!=1800 or saved['probe']['summary']['selected_epoch']!=20 or saved['probe']['summary']['selection']!='fixed_final_epoch':
                raise ValueError('Only fixed final1800/fixedepoch20 models are allowed')
            resources={'encoder':{k:saved['source_encoder'][k]for k in ('path','sha256')},
                'probe':saved['probe']['final_probe'],'summary':record(summary_path),
                'train_cache':saved['feature_caches']['train'],'development_cache':saved['feature_caches']['development']}
            for resource in resources.values():verify(resource)
            arms.append({'seed':seed,'variant':variant,'model_id':model_id,'kind':arm['kind'],
                'config':arm['config'],'resources':resources,
                'encoder_state_hash':saved['probe']['summary']['checkpoint_hash'],
                'probe_state_hash':saved['probe']['summary']['probe_hash']})
    labels=[int(dataset.index[sid]['label'])for sid in fresh]
    sources={**runner.stage_definition_hashes(root),
        'experiments/12_visual_test/evaluation.ipynb':export_digest(root/'experiments/12_visual_test/evaluation.ipynb')}
    protocol={'schema':'notebook12-frozen-reserve-validation/v1','run_id':out.name,
        'source_study':'manifold-long-001','source_plan':record(stage/'plan.json'),
        'source_split':record(source_root/'split_manifest.json'),'source_audit':record(source_root/'duplicate_audit.json'),
        'dataset_manifest':record(root/'data/food101/dataset_manifest.json'),
        'dataset_fingerprint':dataset.fingerprint,'class_names':dataset.class_names,
        'seeds':SEEDS,'variants':VARIANTS,'arms':arms,'source_hashes':sources,
        'sample_ids':fresh,'labels':labels,'membership_hash':digest(fresh),'encoded_images_hash':digest(encoded),
        'observed_development_ids':split['selected_ids']['development'],
        'unused_validation_count':len(fresh),'official_test_count':len(heldout),'official_test':'NOT_RUN',
        'model_selection':'All six final1800 EMA encoders and their fixedepoch20 probes, frozen; no tuning or refitting',
        'preprocessing':'Exactly original deterministic full-image bilinear128 square resize and fixed normalization',
        'endpoint':'Mean of six paired learned-minus-ViT top1 accuracy differences in percentage points',
        'primary_analysis':'Two-sided paired training-seed t test with95%CI,df5; positive mean andCI lower>0 required for superiority; conditional on fresh validation membership',
        'robustness':'Separate two-sided exact seed sign-flip mean test and sign test; crossed seed/image bootstrap exploratory',
        'inference_batch_size':BATCH,'max_run_seconds':CAP_SECONDS,
        'no_optional_stopping':True,'fresh_data_scope':'Previously unused development reserve, not official test; no new independent training seeds',
        'registration_before_fresh_predictions':True,'resume_notebook':'notebooks/12_Visual_Statistical_Validation.ipynb',
        'resume_cell':'run-frozen-validation','checkpoint_directory':str(checkpoint),
        'duplicate_scope':'Original exact decoded RGB duplicate exclusions preserved; near duplicates not ruled out'}
    with run_writer(checkpoint):immutable(out/'protocol.json',protocol)
    return protocol

def _verify_protocol(root,out):
    root,out,checkpoint=_paths(root,out)
    protocol=json.loads((out/'protocol.json').read_text())
    sources={**load_tagged(root,'runner').stage_definition_hashes(root),
        'experiments/12_visual_test/evaluation.ipynb':export_digest(root/'experiments/12_visual_test/evaluation.ipynb')}
    if protocol['source_hashes']!=sources:raise ValueError('Registered exports changed; no silent new run')
    if protocol['run_id']!=out.name or protocol['sample_ids']==protocol['observed_development_ids']:
        raise ValueError('Validation identity changed')
    for key in ('source_plan','source_split','source_audit','dataset_manifest'):verify(protocol[key])
    return protocol

def _dataset(root,protocol):
    manifest=_read_sealed(Path(root)/'data/food101/dataset_manifest.json')
    source=Food101Adapter(Path(root)/'data/food101',manifest)
    if source.fingerprint!=protocol['dataset_fingerprint']:raise ValueError('Dataset fingerprint changed')
    return source

def _probe(arm):
    selected=torch.load(verify(arm['resources']['probe']),map_location='cpu',weights_only=True)
    if selected['selected_epoch']!=20 or selected['selection']!='fixed_final_epoch':raise ValueError('Probe selection changed')
    if encoder_state_hash(selected['state_dict'])!=arm['probe_state_hash'] or selected['encoder_checkpoint_hash']!=arm['encoder_state_hash']:
        raise ValueError('Final probe state identity changed')
    return selected

def _sanity(arm,evaluator):
    saved=json.loads(verify(arm['resources']['summary']).read_text());selected=_probe(arm)
    caches={}
    for split,key in [('train','train_cache'),('development','development_cache')]:
        caches[split]=torch.load(verify(arm['resources'][key]),map_location='cpu',weights_only=True)
        validate_feature_cache(caches[split],caches[split]['identity'])
    result=evaluator._reconstruct_final_probe(selected,caches['train'],caches['development'],saved['probe']['predictions'],saved['probe']['summary'])
    return result

def _encoder(root,arm):
    configure_environment(copy.deepcopy(arm['config']))
    runner=load_tagged(root,'runner');model=runner.build_model(arm['config'],arm['kind']).cpu()
    state=torch.load(verify(arm['resources']['encoder']),map_location='cpu',weights_only=False)
    if state['state']['successful_updates']!=1800 or state['config']!=arm['config']:
        raise ValueError('Final full-state checkpoint update/config changed')
    model.load_state_dict(state['model']);encoder=model.target;del model,state
    if encoder_state_hash(encoder)!=arm['encoder_state_hash']:raise ValueError('EMA encoder identity changed')
    return encoder

def _rng():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
        'cuda':torch.cuda.get_rng_state_all()if torch.cuda.is_available()else None}

def _restore_rng(state):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if state['cuda']is not None:torch.cuda.set_rng_state_all(state['cuda'])

def _progress(folder,identity):
    folder=Path(folder);path=folder/'manifest.json'
    if not path.exists():
        if list(folder.glob('progress-*.pt')):raise ValueError('Orphan progress without manifest; inspect before recovery')
        return None,None
    manifest=json.loads(path.read_text())
    if manifest['identity_hash']!=digest(identity)or not manifest['checkpoints']:raise ValueError('Invalid progress identity')
    for entry in manifest['checkpoints']:
        if Path(entry['file']).name!=entry['file']or digest_file(folder/entry['file'])!=entry['sha256']:
            raise ValueError('Invalid progress checkpoint hash')
    if manifest['latest']!=max(manifest['checkpoints'],key=lambda x:x['next_index'])['file']:raise ValueError('Invalid latest pointer')
    saved=torch.load(folder/manifest['latest'],map_location='cpu',weights_only=False)
    if saved['identity']!=identity or saved['next_index']!=len(saved['sample_ids']) or saved['sample_ids']!=identity['sample_ids'][:saved['next_index']]:
        raise ValueError('Invalid exact feature cursor')
    if len(saved['features'])!=saved['next_index']or len(saved['labels'])!=saved['next_index']or not torch.isfinite(saved['features']).all():
        raise ValueError('Invalid saved feature payload')
    return saved,manifest

def _save_progress(folder,identity,features,labels,ids,timings,complete=False):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    old,manifest=_progress(folder,identity)
    if old is not None and len(ids)<=old['next_index']:raise ValueError('Progress must advance')
    path=folder/f'progress-{len(ids):06d}.pt'
    payload={'schema':'notebook12-frozen-feature-progress/v1','identity':identity,'next_index':len(ids),
        'global_step':math.ceil(len(ids)/BATCH),'features':features,'labels':labels,'sample_ids':ids,'rng':_rng(),
        'model_source':identity['model_source'],'probe_source':identity['probe_source'],
        'optimizer':None,'scheduler':None,'scaler':None,'config':identity['config'],'metrics':{'batch_seconds':timings},
        'distributed':{'world_size':1,'rank':0},'sampler':{'ordered_ids':identity['sample_ids'],'next_index':len(ids)},
        'replay':None,'complete':bool(complete)}
    atomic_torch(path,payload);entry={'file':path.name,'sha256':digest_file(path),'next_index':len(ids)}
    entries=(manifest['checkpoints']if manifest else[])+[entry];kept=sorted(entries,key=lambda x:x['next_index'])[-2:]
    new={'schema':payload['schema'],'identity_hash':digest(identity),'checkpoints':kept,'latest':path.name,
        'next_index':len(ids),'complete':bool(complete),'best':None,
        'best_policy':'Not applicable to fixed frozen inference; no selection',
        'resume_notebook':'notebooks/12_Visual_Statistical_Validation.ipynb','resume_cell':'run-frozen-validation'}
    atomic_json(folder/'manifest.json',new)
    for previous in entries:
        if previous['file']not in {x['file']for x in kept}:(folder/previous['file']).unlink(missing_ok=True)
    return payload

def _extract(root,out,protocol,arm,dataset,ids,folder,deadline=None,stop_after_batches=None):
    identity={'protocol_hash':digest(protocol),'model_id':arm['model_id'],'model_source':arm['resources']['encoder'],
        'probe_source':arm['resources']['probe'],'config':arm['config'],'sample_ids':list(ids),
        'batch_size':BATCH,'preprocessing':preprocessing_descriptor(arm['config']),'pooling':'mean_final_layernorm_patches'}
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);saved,manifest=_progress(folder,identity)
    if saved is not None and saved['complete']:return saved
    encoder=_encoder(root,arm);evaluation=EvaluationDataset(dataset,ids,arm['config']);device=arm['config']['device']
    features=[]if saved is None else[saved['features']];labels=[]if saved is None else[saved['labels']]
    seen=[]if saved is None else list(saved['sample_ids']);timings=[]if saved is None else list(saved['metrics']['batch_seconds'])
    if saved is not None:_restore_rng(saved['rng'])
    count=0;last_commit=time.monotonic()
    try:
        with frozen_encoder(encoder,device),torch.inference_mode():
            for start in range(len(seen),len(ids),BATCH):
                if deadline is not None and time.time()>deadline-3:raise TimeoutError('Original validation deadline exhausted; endpoint remains UNKNOWN')
                tick=time.perf_counter();rows=[evaluation[i]for i in range(start,min(start+BATCH,len(ids)))]
                batch_ids=[x['sample_id']for x in rows]
                if batch_ids!=ids[start:start+len(rows)]:raise ValueError('Input order changed')
                images=torch.stack([x['image']for x in rows]).to(device,dtype=torch.float32)
                tokens=encoder(images)
                if tokens.ndim!=3 or not torch.isfinite(tokens).all():raise FloatingPointError('Nonfinite/unexpected frozen tokens')
                features.append(tokens.mean(1).float().cpu().clone())
                labels.append(torch.tensor([x['label']for x in rows],dtype=torch.int64));seen.extend(batch_ids)
                torch.cuda.synchronize(device);timings.append(time.perf_counter()-tick);count+=1
                del rows,images,tokens
                final=len(seen)==len(ids);limited=stop_after_batches is not None and count>=stop_after_batches
                if count%4==0 or final or limited or time.monotonic()-last_commit>=900:
                    saved=_save_progress(folder,identity,torch.cat(features),torch.cat(labels),seen,timings,complete=final)
                    last_commit=time.monotonic()
                    atomic_json(Path(out)/'status.json',{'status':'RUNNING','arm':arm['model_id'],'images_done':len(seen),'images_total':len(ids),'updated_unix':time.time()})
                if limited:break
    finally:
        encoder.cpu();del encoder;gc.collect();torch.cuda.empty_cache()
    if saved is None:raise RuntimeError('No feature progress committed')
    return saved

def smoke_and_calibrate(root,out):
    """Exact256-image uninterrupted vs128+128 resume for both architectures; no fresh predictions."""
    root,out,checkpoint=_paths(root,out);protocol=_verify_protocol(root,out)
    with run_writer(checkpoint):
        cert_path=out/'smoke_certificate.json'
        if cert_path.exists():
            cert=json.loads(cert_path.read_text())
            if cert['protocol_hash']!=digest(protocol):raise ValueError('Smoke certificate binding changed')
            for rec in cert['checkpoint_records']:verify(rec)
            return cert
        evaluator=load_tagged(root,'evaluation');sanity=[]
        for arm in protocol['arms']:
            result=_sanity(arm,evaluator);sanity.append({'model_id':arm['model_id'],**result})
        dataset=_dataset(root,protocol);ids=protocol['observed_development_ids'][:256];results=[];records=[]
        original=_rng()
        try:
            for variant in VARIANTS:
                arm=next(x for x in protocol['arms']if x['seed']==SEEDS[0]and x['variant']==variant)
                rng=copy.deepcopy(original)
                _restore_rng(rng)
                reference=_extract(root,out,protocol,arm,dataset,ids,checkpoint/'smoke'/variant/'reference')
                _restore_rng(rng)
                partial=_extract(root,out,protocol,arm,dataset,ids,checkpoint/'smoke'/variant/'split',stop_after_batches=2)
                if partial['next_index']not in (128,256):raise AssertionError('Smoke split cursor differs')
                resumed=_extract(root,out,protocol,arm,dataset,ids,checkpoint/'smoke'/variant/'split')
                for key in ('features','labels'):
                    if not torch.equal(reference[key],resumed[key]):raise AssertionError('Exact feature resume differs: '+key)
                if reference['sample_ids']!=resumed['sample_ids']or reference['next_index']!=256:
                    raise AssertionError('Exact sample cursor differs')
                old_cache=torch.load(verify(arm['resources']['development_cache']),map_location='cpu',weights_only=True)
                if old_cache['sample_ids'][:256]!=ids:raise ValueError('Smoke original development order differs')
                torch.testing.assert_close(reference['features'],old_cache['features'][:256],rtol=0,atol=0)
                probe=_probe(arm)
                def logits(saved):
                    x=(saved['features']-probe['standardization']['mean'])/probe['standardization']['std']
                    return F.linear(x,probe['state_dict']['weight'],probe['state_dict']['bias'])
                if not torch.equal(logits(reference),logits(resumed)):raise AssertionError('Resume predictions differ')
                raw=reference['metrics']['batch_seconds'][1:]
                results.append({'variant':variant,'status':'PASS','exact_features':True,'exact_original_cache_features':True,
                    'exact_predictions':True,'reference_images':256,'resume_at_images':128,
                    'median_batch_seconds':float(np.median(raw)),'raw_measured_batch_seconds':raw})
                for mode in ('reference','split'):
                    manifest=json.loads((checkpoint/'smoke'/variant/mode/'manifest.json').read_text())
                    records.append(record(checkpoint/'smoke'/variant/mode/manifest['latest']))
        finally:_restore_rng(original)
        estimate=sum(x['median_batch_seconds']for x in results)*math.ceil(len(protocol['sample_ids'])/BATCH)*len(SEEDS)*1.35+180
        cert={'status':'PASS','protocol_hash':digest(protocol),'arms':results,'prior_development_reconstruction':sanity,
            'checkpoint_records':records,'estimated_full_inference_seconds':estimate,'max_run_seconds':CAP_SECONDS,
            'timing_scope':'Same frozen models and exact preprocessing on observed development; excludes setup;1.35margin+180seconds',
            'fresh_images_used':0,'official_test_images_used':0,'training_updates':0}
        if estimate>CAP_SECONDS:raise RuntimeError('Calibrated complete12arm inference exceeds45minutecap')
        immutable(out/'smoke_certificate.json',cert);immutable(out/'calibration.json',cert)
        return cert

def run_validation(root,out,deadline_unix=None):
    """Run all12 frozen arms on8074 fresh reserve images, resuming exact committed cursors.
    Original45minute deadline is persisted once and never extends on restart.
    """
    root,out,checkpoint=_paths(root,out);protocol=_verify_protocol(root,out)
    smoke=json.loads((out/'smoke_certificate.json').read_text())
    if smoke['status']!='PASS'or smoke['protocol_hash']!=digest(protocol):raise ValueError('Exact smoke must pass before full validation')
    with run_writer(checkpoint):
        final_path=out/'predictions.json'
        if final_path.exists():
            final=json.loads(final_path.read_text())
            if final['registration_sha256']!=digest_file(out/'protocol.json')or digest_file(final['npz_path'])!=final['npz_sha256']:
                raise ValueError('Completed predictions provenance changed')
            for item in final['arms']:
                if digest_file(item['predictions_csv'])!=item['predictions_csv_sha256']:raise ValueError('Completed arm CSV changed')
            return final
        clock_path=checkpoint/'clock.json'
        if clock_path.exists():clock=json.loads(clock_path.read_text())
        else:
            now=time.time();deadline=now+CAP_SECONDS
            if deadline_unix is not None:deadline=min(deadline,float(deadline_unix))
            clock={'started_unix':now,'deadline_unix':deadline,'protocol_hash':digest(protocol)};atomic_json(clock_path,clock)
        if clock['protocol_hash']!=digest(protocol):raise ValueError('Original validation clock identity changed')
        deadline=clock['deadline_unix'];dataset=_dataset(root,protocol);arms=[];predictions=np.empty((6,2,len(protocol['sample_ids'])),dtype=np.int64)
        try:
            for i,seed in enumerate(SEEDS):
                for j,variant in enumerate(VARIANTS):
                    arm=next(x for x in protocol['arms']if x['seed']==seed and x['variant']==variant)
                    _sanity(arm,load_tagged(root,'evaluation'))
                    saved=_extract(root,out,protocol,arm,dataset,protocol['sample_ids'],checkpoint/'features'/arm['model_id'],deadline=deadline)
                    if not saved['complete']:raise ValueError('Only complete frozen arm outputs enter final endpoint')
                    if saved['labels'].tolist()!=protocol['labels']:raise ValueError('Source labels/order changed')
                    selected=_probe(arm);x=(saved['features']-selected['standardization']['mean'])/selected['standardization']['std']
                    with torch.inference_mode():
                        logits=torch.cat([F.linear(batch,selected['state_dict']['weight'],selected['state_dict']['bias'])for batch in x.split(256)])
                    metrics=classification_metrics(logits,saved['labels'])
                    rows=prediction_rows(logits,saved['labels'],saved['sample_ids'],arm['model_id'],'fresh_development_reserve',
                        arm['encoder_state_hash'],arm['probe_state_hash'],seed)
                    for row in rows:row['variant']=variant
                    csv_path=out/'predictions'/(arm['model_id']+'.csv');write_csv(csv_path,rows)
                    predictions[i,j]=logits.argmax(1).numpy()
                    arms.append({'seed':seed,'variant':variant,'model_id':arm['model_id'],'predictions_csv':str(csv_path),
                        'predictions_csv_sha256':digest_file(csv_path),'top1_percent':100*metrics['top1'],'metrics':metrics,
                        'encoder_state_hash':arm['encoder_state_hash'],'probe_state_hash':arm['probe_state_hash'],
                        'feature_checkpoint_manifest':record(checkpoint/'features'/arm['model_id']/'manifest.json')})
                    atomic_json(out/'status.json',{'status':'RUNNING','completed_arms':len(arms),'total_arms':12,
                        'updated_unix':time.time(),'deadline_unix':deadline,'official_test':'NOT_RUN'})
                    print('Frozen validation',arm['model_id'],len(saved['sample_ids']),'images complete',flush=True)
            npz_path=out/'predictions.npz';temp=out/(npz_path.name+'.'+uuid.uuid4().hex+'.tmp')
            with temp.open('wb')as stream:
                np.savez_compressed(stream,predictions=predictions,labels=np.asarray(protocol['labels'],dtype=np.int64),
                    sample_ids=np.asarray(protocol['sample_ids']),seeds=np.asarray(SEEDS),variants=np.asarray(VARIANTS));stream.flush();os.fsync(stream.fileno())
            os.replace(temp,npz_path)
            final={'schema':'notebook12-frozen-predictions/v1','status':'COMPLETE','fresh_status':'COMPLETE','run_id':out.name,
                'seeds':SEEDS,'variants':VARIANTS,'sample_ids':protocol['sample_ids'],'labels':protocol['labels'],
                'class_names':protocol['class_names'],'source_hashes':protocol['source_hashes'],'npz_path':str(npz_path),'npz_sha256':digest_file(npz_path),
                'registration_sha256':digest_file(out/'protocol.json'),'protocol_path':str(out/'protocol.json'),'arms':arms,
                'source_study':'manifold-long-001','official_test':'NOT_RUN','new_training_seeds':0,
                'fresh_images':len(protocol['sample_ids']),'started_unix':clock['started_unix'],'completed_unix':time.time(),
                'elapsed_seconds':time.time()-clock['started_unix'],'deadline_unix':deadline,'training_updates':0}
            atomic_json(final_path,final);atomic_json(out/'status.json',{'status':'COMPLETE','completed_arms':12,'total_arms':12,
                'updated_unix':time.time(),'elapsed_seconds':final['elapsed_seconds'],'official_test':'NOT_RUN'})
            return final
        except TimeoutError as error:
            atomic_json(out/'status.json',{'status':'UNKNOWN_CUTOFF','reason':str(error),'updated_unix':time.time(),
                'deadline_unix':deadline,'official_test':'NOT_RUN'})
            raise
